# I2C device initialization

In [ ]:
from time import sleep as delay
from project.sc8583 import project

if "ic" in globals():
    ic.close()
    
ic = project(device="sc8583", revision="1p1", emulator="ch341", logging=False, is_gui=False)

# Equipments initialization

In [ ]:
from phy.multimeter import keithley_dm6500
from phy.power_supply import rigol_dp821a, rigol_dp811a
from phy.power_analyzer import keysight_N6705
from phy.scope import tektronix_mdo34
from phy.battery_simulator import asd_906b

# from phy.eloader import sdl1030x
# from phy.relay_16ch import relay_box

%matplotlib tk
from interface.docs.output_chart import plot
from interface.cui_colors import color
from interface.i2c_bridge.tca9548a import tca9548
from interface.docs.output_excel import excel_frame, style
from interface.cui_logger import logger as log

import numpy as np
from time import sleep as delay
chart = plot()

dm1 = keithley_dm6500()
dm2 = keithley_dm6500("USB0::0x05E6::0x6500::04651251::INSTR")
# ps1 = rigol_dp821a()
# ps2 = rigol_dp811a()
ps = keysight_N6705()
ds = tektronix_mdo34()
bs = asd_906b(port=7)
# sm = keithley_2470()
# ld = sdl1030x()

# relay = relay_box(i2c_h=ic)
# tc = chamber(port=3)
# mux = tca9548(ic.i2c_h, 0x70)

# --------------------------------------------------
# list_voltage = list(np.arange(5, 8, 0.005))
# voltage  = [round(num, 3) for num in list_voltage]
# --------------------------------------------------

from concurrent.futures import ThreadPoolExecutor

# Test code block

In [ ]:
%matplotlib tk
from interface.docs.output_chart import plot
import numpy as np
import math
chart = plot()

In [ ]:
# absolute maximum rating

#-------------------#
max_rating = 36
item = "VEXT_DRV"
#-------------------#

temp = list(np.arange(5, max_rating+0.1, 0.5))
voltage_sweep = [round(num, 1) for num in temp]

ps.ch1.cfg_all = 5, 0.01
ps.ch2.cfg_all = 4, 0.01

In [ ]:
chart.set_title = item

chart.set_single_plot
chart.set_xlabel = "Voltage (V)"
chart.set_y_main_label = "Current (uA)"

chart.add_main_item = "PIN"

chart.PIN.set_linewidth = 2
chart.PIN.set_line_dot

In [ ]:
ps.ch1.enable
delay(0.5)

for _ in range(3):
    dm1.current_1E_3

for set_volt in voltage_sweep:

    ps.ch1.vset = set_volt
    meas_i = dm1.current_1E_3 * 1e+6

    if meas_i > 5000:
        for _ in range(3):
            print("test")
            meas_i = dm1.current_1E_3 * 1e+6


    chart.PIN.set_data = set_volt, meas_i
    chart.output

    print(f"{set_volt:.01f}, {meas_i:.06f}")

chart.save_plot = chart.set_title
ps.ch1.disable

In [ ]:
# for VEXT_DRV

ps.ch1.enable
ps.ch2.enable
delay(0.5)

ic.VEXT_MANUAL_EN = 1
ic.VEXT_DRV_ON = 1
ic.VEXT_OVP_DIS = 1
ic.VIN_OVP_DIS = 1

print(f"VEXT_MANUAL_EN : {ic.VEXT_MANUAL_EN}")
print(f"VEXT_DRV_ON : {ic.VEXT_DRV_ON}")

for _ in range(3):
    dm1.current_1E_3

for set_volt in voltage_sweep:

    ps.ch1.vset = set_volt
    ps.ch2.vset = set_volt - 1

    meas_i = dm1.current_1E_3 * 1e+6

    if meas_i > 5000:
        for _ in range(2):
            print("test")
            meas_i = dm1.current_1E_3 * 1e+6


    chart.PIN.set_data = set_volt, meas_i
    chart.output

    print(f"{set_volt:.01f}, {meas_i:.06f}")

    if meas_i > 5000:
        break

chart.save_plot = chart.set_title

for rev in reversed(voltage_sweep):
    ps.ch1.vset = rev
    ps.ch2.vset = rev - 1
    print(rev)

ps.ch1.disable
ps.ch2.disable

In [ ]:
temp = list(np.arange(3, 20.1, 1))
voltage_sweep = [round(num, 1) for num in temp]

ps.ch2.cfg_all = voltage_sweep[0], 0.05
ps.ch2.enable

ic.VEXT_OVP_DIS = 1

In [ ]:
for vext in voltage_sweep:
    ps.ch2.vset = vext
    i_vext = round(dm2.current_100E_3 * 1000_000,3)
    i_vout = round(dm1.current_10E_3 * 1000_000,3)

    print(vext, i_vext, i_vout)

    if i_vext > 9_000_000:
        break

    if i_vout > 9_000_000:
        break

ps.ch2.disable
ps.ch2.vset = voltage_sweep[0], 0.05